# 10 — Calibración del modelo final (Persona B)

**Pregunta:** cuando el modelo dice "40 % de que gane el local", ¿el local gana más o menos 4 de cada 10 veces?
Eso es estar **bien calibrado**. Para un proyecto cuyo producto son probabilidades, es más importante que
acertar el resultado más probable.

Se comparan, en los **mismos 380 partidos de validación (2023-24)**:
- el modelo final `logistica_sin_redundancia` (`FEATURES_FINAL`), **entrenado solo con train**;
- `logistica_elo` (solo Elo), el modelo de comparación;
- las cuotas del mercado.

**Herramientas** (`src/evaluation/calibracion.py`):
- `datos_calibracion(y, probs, clase)`: ordena los partidos según la probabilidad predicha para una clase y los
  agrupa en 10 bins **por cuantiles** (38 partidos cada uno, ninguno vacío). En cada bin compara la
  probabilidad media predicha con la frecuencia con que de verdad ocurrió.
- `ece(y, probs)`: *error de calibración esperado*, la distancia media entre ambas (ponderada por partidos),
  promediada sobre H, D y A. 0 = perfecto.
- `ece_por_azar(probs)`: cuánto ECE saldría **solo por azar** si el modelo fuera perfecto. Con 38 partidos por
  bin, la frecuencia observada oscila varios puntos aunque la probabilidad sea exacta, así que el ECE nunca es 0.

In [ ]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd

from src.config import CLASSES
from src.splits import split
from src.models.final import FEATURES_FINAL
from src.models.logistic import tune_logistic
from src.models.common import proba_hda
from src.evaluation.metrics import evaluate, implied_probs
from src.evaluation.calibracion import datos_calibracion, ece, ece_por_clase, ece_por_azar
from src.evaluation.figuras import fig_calibracion

df = pd.read_csv("../data/processed/features.csv", parse_dates=["date"])
train, val = split(df)                                  # sin use_test: el test no se toca
assert train["date"].max() < val["date"].min()
val_ok = val[val[["odds_H", "odds_D", "odds_A"]].notna().all(axis=1)]
y = val_ok["result"]

modelos = {"logistica_sin_redundancia": FEATURES_FINAL, "logistica_elo": ["elo_diff"]}
probs = {nombre: proba_hda(tune_logistic(train, cols).best_estimator_, val_ok[cols])   # SOLO train
         for nombre, cols in modelos.items()}
probs["cuotas_mercado"] = implied_probs(val_ok)
print(f"train: {len(train)} | val con cuotas: {len(val_ok)}")
for nombre, p in probs.items():
    print(f"{nombre:<27} log loss = {evaluate(y, p, nombre)['log_loss']:.4f}")

## 1. Primer control: ¿acierta las proporciones globales?

Si el modelo dice en promedio "45 % local", en la temporada debería ganar el local alrededor del 45 % de los partidos.

In [ ]:
medias = pd.DataFrame({nombre: p.mean(axis=0) for nombre, p in probs.items()}, index=CLASSES).T
medias.loc["frecuencia real"] = y.value_counts(normalize=True).reindex(CLASSES).to_numpy()
medias.round(3)

## 2. Datos de calibración, bin por bin (modelo final, victoria local)

Cada fila son 38 partidos. Bien calibrado = `prob_media` ≈ `frec_observada` en todas las filas.

In [ ]:
datos_calibracion(y, probs["logistica_sin_redundancia"], "H").round(3)

## 3. ECE de cada modelo y ECE esperable por azar

`ece_azar_p50` y `ece_azar_p95`: mediana y percentil 95 del ECE de un modelo **perfecto** que diera esas
mismas probabilidades (500 simulaciones, semilla fija). Si el ECE real queda por debajo del p95, la
diferencia con la diagonal es compatible con el azar.

In [ ]:
filas = []
for nombre, p in probs.items():
    azar = ece_por_azar(p)
    filas.append({"modelo": nombre, **{f"ece_{c}": v for c, v in ece_por_clase(y, p).items()},
                  "ece": ece(y, p), "ece_azar_p50": np.median(azar), "ece_azar_p95": np.quantile(azar, 0.95),
                  "log_loss": evaluate(y, p, nombre)["log_loss"]})
tabla_ece = pd.DataFrame(filas).sort_values("ece").reset_index(drop=True)
tabla_ece.to_csv("../docs/calibracion_validacion.csv", index=False)
tabla_ece.round(4)

## 4. Curvas de calibración: modelo final vs cuotas

In [ ]:
fig = fig_calibracion(y, {"Modelo final": probs["logistica_sin_redundancia"],
                          "Cuotas del mercado": probs["cuotas_mercado"]},
                      "../docs/fig_calibracion.png")

## 5. ¿Hace falta recalibrar?

Regla acordada: solo se intenta calibrar (sigmoid, usando únicamente la última temporada de train como conjunto
de calibración) si el modelo final está **claramente mal calibrado**, es decir, si su ECE es mucho peor que el de
las cuotas. Aquí lo concretamos así: el ECE del modelo final supera el p95 del azar **y** el de las cuotas
en más de 0.02.

In [ ]:
fila = tabla_ece.set_index("modelo")
ece_final, ece_cuotas = fila.loc["logistica_sin_redundancia", "ece"], fila.loc["cuotas_mercado", "ece"]
p95_final = fila.loc["logistica_sin_redundancia", "ece_azar_p95"]
mal_calibrado = ece_final > p95_final and ece_final - ece_cuotas > 0.02
print(f"ECE final {ece_final:.4f} | p95 por azar {p95_final:.4f} | ECE cuotas {ece_cuotas:.4f} "
      f"| diferencia {ece_final - ece_cuotas:+.4f}")
print("¿Claramente mal calibrado?", mal_calibrado)

## 6. Conclusión

| Modelo | ECE H | ECE D | ECE A | **ECE** | ECE de un modelo perfecto (p50 / p95) | Log loss |
|---|---|---|---|---|---|---|
| Cuotas del mercado | 0.053 | 0.033 | 0.057 | **0.048** | 0.055 / 0.069 | 0.909 |
| `logistica_elo` | 0.058 | 0.052 | 0.036 | **0.049** | 0.055 / 0.072 | 0.932 |
| `logistica_sin_redundancia` (final) | 0.058 | 0.055 | 0.053 | **0.056** | 0.055 / 0.070 | 0.926 |

- **El modelo final está bien calibrado.** Su ECE (0.056) es prácticamente igual a la mediana que tendría un
  modelo perfecto con 380 partidos (0.055) y queda lejos del percentil 95 (0.070). Las diferencias que se ven
  con la diagonal son del tamaño que produce el azar con 38 partidos por punto.
- **También acierta las proporciones globales:** predice en promedio 45.5 % local, 21.8 % empate y 32.7 % visitante,
  y en la temporada hubo 46.1 %, 21.6 % y 32.4 %.
- **Las cuotas están un poco mejor calibradas** (0.048), pero la diferencia (+0.008) es pequeña y las dos están
  dentro del ruido. Las curvas del modelo y del mercado van casi juntas en los tres paneles. Lo que separa al
  mercado de nuestro modelo es la **resolución** (distingue mejor un partido fácil de uno difícil), no la calibración.
- **Empates:** el modelo nunca da más de 31 % de empate (rango 9 %–31 %), igual que las cuotas (8 %–30 %).
  El empate es el resultado más difícil de anticipar y tanto el modelo como el mercado lo comprimen en un
  rango estrecho; por eso el panel D ocupa solo una franja pequeña del gráfico.
- **Recalibrar no hizo falta.** El criterio (ECE mucho peor que las cuotas) no se cumple, así que **no se probó**
  la calibración sigmoid. Además, recalibrar un modelo que ya está en el nivel del azar suele empeorar el log loss,
  porque el calibrador ajusta ruido. **El modelo de la app no cambia.**
- **Limitación:** con 380 partidos, el ECE solo detecta errores grandes (más de ~7 puntos). La revisión final
  será en el test 2024-25.
